# 4D SfM — DEM + DoD + M3C2 raster (monthly batch)

Runs the full per-date workflow for one date per month, producing
DEM + orthoimage + DoD + stable-terrain DoD + M3C2 raster (with
histograms) for each date. All logic lives in
`cntp.pipeline_4dsfm.run_4dsfm_day_with_rasters`; this notebook is
just configuration + one loop.

Reference rasters (`reference_dem.tif`, `reference_ortho.tif`,
`reference_dem_stable.tif`) cache in `_ref_cache/` — built on the
first iteration, skipped on every subsequent date.

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"

from pathlib import Path

import Metashape  # noqa: F401  — must import after AGISOFT_LICENSE_PATH is set
from cntp.pipeline_4dsfm import run_4dsfm_day_with_rasters

## Configuration

Edit only this section. Paths + dates + all per-stage knobs live here.

In [2]:
# ── Paths ────────────────────────────────────────────────────────────
base_dir     = Path("/mnt/g/2023_11_Nepal/2023_Changri")
tlcam_dir    = base_dir / "TLCAM"/"ChangriWest_renamed"
output_dir   = base_dir

ref_cloud    = base_dir / "Ref_PC" / "Reference_UAV_TLC_PCS.laz"
glacier_mask = base_dir / "glaciermask_new" / "glacier_mask_pcs.shp"
registry_csv = output_dir / "output_new" / "reference_registry.csv"

# ── Dates to process (one per month) ─────────────────────────────────
monthly_dates = [
    "2024-01-18",
    "2024-02-18",
    "2024-03-17",
    "2024-04-19",
    "2024-05-17",
    "2024-06-15",
    "2024-07-15",
    "2024-08-17",
    "2024-09-18",
    "2024-10-16",
    "2024-11-15",
]

# ── Per-date knobs (SfM + raster combined) ───────────────────────────
params = dict(
    # SfM pipeline knobs (forwarded to run_4dsfm_day)
    match_downscale       = 1,
    depth_downscale       = 2,
    loc_acc_new           = (0.5, 0.5, 0.5),
    rot_acc_new           = (5.0, 5.0, 5.0),
    ref_downsample        = 0.4,
    tba_downsample        = 1.0,
    p2p_max_disp          = 10.0,
    sp2p_max_disp         =  5.0,
    m_sp2p_max_disp       =  0.5,
    use_ecef              = True,
    overwrite             = False,
    verbose               = True,
    # When False, day's cameras are NOT appended to the registry —
    # keeps it frozen at the original 2023-11-27 baseline across the
    # whole batch (we may flip this after supervisor sign-off).
    add_to_registry       = False,

    # Raster knobs
    res                   = 1.0,
    max_gap_pixels        = 1,
    ref_cloud_downsample  = 0.25,
    m3c2_ref_downsample   = 0.25,
    slope_threshold       = 60.0,
    overwrite_ref_dem     = False,
    overwrite_day_dem     = False,
    overwrite_dod         = False,
    overwrite_stable      = False,
    overwrite_stable_dod  = False,
    overwrite_m3c2        = False,
)

## Run

Loops over `monthly_dates` and calls `run_4dsfm_day_with_rasters`
for each. Wrapped in `try / except` so a single bad date doesn't
stop the run; per-date stats collected and printed at the end.

In [3]:
import traceback

summary = []
for d in monthly_dates:
    try:
        summary.append(run_4dsfm_day_with_rasters(
            new_date     = d,
            tlcam_dir    = tlcam_dir,
            ref_cloud    = ref_cloud,
            glacier_mask = glacier_mask,
            registry_csv = registry_csv,
            output_dir   = output_dir,
            **params,
        ))
    except Exception as e:
        print(f"\n  ⚠ {d} failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        summary.append({"date": d, "error": repr(e)})

print(f"\n{'='*70}\n  Batch summary ({len(summary)} dates)\n{'='*70}")
for s in summary:
    if "error" in s:
        print(f"  {s['date']} : ERROR — {s['error']}")
    else:
        print(
            f"  {s['date']} : "
            f"DoD med={s['dod_stats']['median']:+.3f} m  std={s['dod_stats']['std']:.3f}  |  "
            f"stable med={s['stable_stats']['median']:+.3f} m  std={s['stable_stats']['std']:.3f}  |  "
            f"M3C2 med={s['m3c2_stats']['median']:+.3f} m  std={s['m3c2_stats']['std']:.3f}"
        )

[Step 1] Skipping — 2024-01-18_cameras_4DSfM.csv exists
[Step 2] Skipping — 2024-01-18_cloud.las exists
  Stable reference cached → Reference_UAV_TLC_PCS_ds0.40_stable.las
[Step 3] Skipping — 2024-01-18_cloud_coreg_hsfm.las exists
[Step 3b] Skipping — stable TBA exists
[Step 4] Skipping — 2024-01-18_cameras_coreg.csv exists
[Step 6] Skipping — 2024-01-18_cloud_validated.laz exists
[Step 6b] Skipping — 2024-01-18_cloud_validated_stable.laz exists

[Step 7] Skipping — add_to_registry=False (registry kept frozen)
  DEM + ortho cached → reference_dem.tif, reference_ortho.tif
  DEM + ortho cached → 2024-01-18_dem.tif, 2024-01-18_ortho.tif
  DoD cached → DOD.tif
  Stable DEM cached → reference_dem_stable.tif
  Stable DEM cached → 2024-01-18_dem_stable.tif
  DoD cached → DOD_stable.tif
  M3C2 raster cached → M3C2_raster.tif

  [2024-01-18] DoD med=-0.194 m  |  stable med=-0.494 m  |  M3C2 med=-0.146 m
[Step 1] Skipping — 2024-02-18_cameras_4DSfM.csv exists
[Step 2] Skipping — 2024-02-18_cloud

Can't load OpenCL library
failed to enable cpu vulkan support (/home/asus/miniconda3/envs/cntp/bin/lib dir not exists)


Found 1 GPUs in 0.295608 sec (CUDA: 0.072378 sec, OpenCL: 0.000339 sec, Vulkan: 0.222851 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]
[GPU] photo 0: 80000 points
[GPU] photo 20: 80000 points
[GPU] photo 24: 80000 points
[GPU] photo 28: 80000 points
[GPU] photo 7: 80000 points
[GPU] photo 32: 80000 points
[GPU] photo 36: 80000 points
[GPU] photo 10: 80000 points
[GPU] photo 40: 80000 points
[GPU] photo 44: 80000 points
[GPU] photo 13: 80000 points
[GPU] photo 48: 80000 points
[GPU] photo 52: 80000 points
[GPU] photo 16: 80000 points
[GPU] photo 56: 80000 points
[GPU] photo 60: 80000 points
[GPU] photo 64: 80000 points
points detected in 8.78192 sec
loaded object list in 0.002756 sec
loaded keypoint partition in 0.003274 sec
loaded matching data in 0.002375 sec
Found 1 GPUs in 0.000571 sec (CUDA: 1.7e-05 sec,

Can't load OpenCL library


[GPU] photo 1: 80000 points
[GPU] photo 21: 80000 points
[GPU] photo 25: 80000 points
[GPU] photo 4: 80000 points
[GPU] photo 29: 80000 points
[GPU] photo 33: 80000 points
[GPU] photo 37: 80000 points
[GPU] photo 11: 80000 points
[GPU] photo 41: 80000 points
[GPU] photo 45: 80000 points
[GPU] photo 14: 80000 points
[GPU] photo 49: 80000 points
[GPU] photo 53: 80000 points
[GPU] photo 17: 80000 points
[GPU] photo 57: 80000 points
[GPU] photo 61: 80000 points
points detected in 8.07823 sec
loaded object list in 0.002314 sec
loaded keypoint partition in 0.002562 sec
loaded matching data in 0.00211 sec
Found 1 GPUs in 0.000658 sec (CUDA: 1.8e-05 sec, OpenCL: 0.000315 sec, Vulkan: 0.000306 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]


...

Found 1 GPUs in 0.055842 sec (CUDA: 0.003873 sec, OpenCL: 0.046317 sec, Vulkan: 0.005288 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]
group 1/1: cameras images prepared in 8.10836 s
group 1/1: 45 x frame
group 1/1: 45 x uint8
group 1/1: expected peak VRAM usage: 1042 MB (404 MB max alloc, 6512x7326 mipmap texture, 16 max neighbors)
Found 1 GPUs in 0.003156 sec (CUDA: 2.4e-05 sec, OpenCL: 0.002348 sec, Vulkan: 0.000756 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]
Using device 'NVIDIA GeForce RTX 3050 Ti Laptop GPU' in concurrent. (2 times)
[GPU 1] group 1/1: estimating depth map for 1/45 camera 0 (5 neighbs)...
[GPU 2] group 1/1:

Can't load OpenCL library


[GPU 1] Camera 0 samples after final filtering: 46% (1.75967 avg inliers) = 100% - 1% (not matched) - 17% (bad matched) - 2% (no neighbors) - 9% (no cost neighbors) - 14% (inconsistent normal) - 0% (estimated bad angle) - 0% (found bad angle) - 10% (speckles filtering)
[GPU 1] Camera 0 tile #1/4: level #6/6 (x2 downscale: 1664x1280, image blowup: 3328x2560) done in 1.39005 s = 44% propagation + 37% refinement + 10% filtering + 0% smoothing
Peak VRAM usage updated: Camera 0 (5 neihbs): 241 MB = 108 MB gpu_neighbImages (45%) + 28 MB gpu_mipmapNeighbImage (12%) + 20 MB gpu_tmp_hypo_ni_cost (8%) + 12 MB gpu_tmp_normal (5%) + 10 MB gpu_neighbMasks (4%) + 8 MB gpu_refImage (3%) + 8 MB gpu_depth_map (3%) + 8 MB gpu_cost_map (3%) + 8 MB gpu_coarse_depth_map_radius (3%) + 8 MB gpu_coarse_depth_map (3%)
[GPU 2] Camera 1 samples after final filtering: 35% (1.66792 avg inliers) = 100% - 3% (not matched) - 23% (bad matched) - 4% (no neighbors) - 11% (no cost neighbors) - 14% (inconsistent normal) -

Can't load OpenCL library


Camera 0 (5 neighbs) level #1/3 filtering: 9% good (5% of speckles) + 11% norm (95% of speckles) - 0% speckles + 3% bad + 76% empty (37% inliers support + 1% inliers intersects + 1% inliers doesn't reach + 4% inliers no depth + 53% outliers support + 3% outliers intersects + 3% outliers doesn't reach + 1% inliers occludes + 2% outliers occludes)
Camera 0 (5 neighbs) level #2/3 filtering: 16% good (3% of speckles) + 14% norm (97% of speckles) - 0% speckles + 5% bad + 65% empty (0% inliers support + 0% inliers intersects + 0% inliers doesn't reach + 0% inliers no depth + 97% outliers support + 3% outliers intersects + 2% outliers doesn't reach + 0% inliers occludes + 9% outliers occludes)
Camera 0 (5 neighbs) level #3/3 filtering: 23% good (5% of speckles) + 11% norm (95% of speckles) - 0% speckles + 4% bad + 62% empty (0% inliers support + 0% inliers intersects + 0% inliers doesn't reach + 0% inliers no depth + 99% outliers support + 1% outliers intersects + 1% outliers doesn't reach + 